# 15-minute data detective: prototype car drag

**Engineering context:** Your team has completed wind-tunnel tests on a prototype car. The eventual goal is to predict the **drag coefficient** (`drag_coefficient`) from test conditions and design settings. Before fitting any machine-learning model, you must decide whether the measurements are trustworthy.

**Your mission:** Load the data, visualise it, and find as many *strange things* as you can. For each issue, decide: **correct it, remove it, or ask the test engineer?**

Suggested timing: 2 min load and inspect · 6 min visualise · 5 min investigate · 2 min reveal/discuss.

**ERRORS: If the cells do not execute due to missing libraries, please open a terminal from anaconda navigator and install the library using pip. (e.g. pip install pandas)**

## Data dictionary

| Column | Meaning | Typical expectation |
|---|---|---|
| `test_id` | Wind-tunnel run identifier | Unique |
| `wind_speed_m_s` | Tunnel air speed | 15–70 m/s |
| `yaw_angle_deg` | Car angle to airflow | About −10° to +10° |
| `ride_height_mm` | Chassis height above road | 70–200 mm |
| `rear_spoiler_deg` | Spoiler setting | 0–25° |
| `air_temp_c` | Tunnel air temperature | 10–35 °C |
| `air_density_kg_m3` | Air density | 0.9–1.4 kg/m³ |
| `drag_coefficient` | Prediction target, $C_d$ | Usually 0.20–0.80 for this vehicle |

In [ ]:
# Run this cell first. It uses only packages included with Anaconda.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

DATA_FILE = Path("data/car_drag_wind_tunnel_measurements.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f'{DATA_FILE} was not found. Put the CSV in the same folder as this notebook.'
    )

df = pd.read_csv(DATA_FILE)
print(f'Loaded {df.shape[0]} rows and {df.shape[1]} columns')
df.head()

FileNotFoundError: data/car_drag_wind_tunnel_measurements.csv was not found. Put the CSV in the same folder as this notebook.

## 1 — First inspection (2 minutes)

Run the next cell. In pairs, note anything unexpected about the size, column types, missing values, or repeated records. **Do not clean anything yet.**

In [ ]:
print('COLUMN TYPES')
display(df.dtypes.to_frame('dtype'))

print('MISSING VALUES')
display(df.isna().sum().to_frame('missing_count'))

print(f'Exact duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate test IDs: {df["test_id"].duplicated().sum()}')
display(df[df.duplicated(keep=False)])

## 2 — Visual investigation (6 minutes)

Before running the plots, predict which inputs should affect drag. Then look for:

- values far away from the rest;
- breaks in otherwise sensible relationships;
- impossible values or possible unit/decimal errors;
- measurements missing from a plot.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].scatter(df['rear_spoiler_deg'], df['drag_coefficient'], alpha=0.8)
axes[0, 0].set(xlabel='Rear spoiler angle (deg)', ylabel='Drag coefficient',
               title='Does spoiler angle affect drag?')

points = axes[0, 1].scatter(df['wind_speed_m_s'], df['drag_coefficient'],
                            c=df['rear_spoiler_deg'], cmap='viridis', alpha=0.85)
axes[0, 1].set(xlabel='Wind speed (m/s)', ylabel='Drag coefficient',
               title='Speed vs drag coefficient')
fig.colorbar(points, ax=axes[0, 1], label='Spoiler angle (deg)')

axes[1, 0].scatter(df['ride_height_mm'], df['drag_coefficient'], alpha=0.8, color='tab:orange')
axes[1, 0].set(xlabel='Ride height (mm)', ylabel='Drag coefficient',
               title='Ride height vs drag coefficient')

df['drag_coefficient'].plot(kind='box', ax=axes[1, 1], vert=False)
axes[1, 1].set(title='Drag coefficient: box plot', xlabel='Drag coefficient')

plt.tight_layout()
plt.show()

## 3 — Investigate suspicious records (5 minutes)

Summary statistics often expose issues hidden by a plot scale. Run the next cells and answer:

1. Which records would you query with the test engineer?
2. Is each anomaly a statistical outlier, a physically impossible value, or both?
3. Why could deleting every outlier be a bad engineering decision?

In [ ]:
# Compare ranges and quartiles. Rotate the table to make it easier to scan.
display(df.describe().T)

# Show the most extreme rows for selected engineering variables.
for column in ['wind_speed_m_s', 'ride_height_mm',
               'air_density_kg_m3', 'drag_coefficient']:
    print(f'\nExtremes for {column}')
    display(df.nsmallest(2, column)[['test_id', column]])
    display(df.nlargest(2, column)[['test_id', column]])

In [ ]:
# Optional challenge: use the engineering limits from the data dictionary.
outside_limits = (
    ~df['wind_speed_m_s'].between(15, 70)
    | ~df['yaw_angle_deg'].between(-10, 10)
    | ~df['ride_height_mm'].between(70, 200)
    | ~df['rear_spoiler_deg'].between(0, 25)
    | ~df['air_temp_c'].between(10, 35)
    | ~df['air_density_kg_m3'].between(0.9, 1.4)
    | ~df['drag_coefficient'].between(0.20, 0.80)
)

display(df.loc[outside_limits])

## Stop and report

Write a one-sentence handover to the aerodynamics lead:

> I would **not start modelling yet** because ...

Only then open the reveal below.

<details>
<summary><strong>Reveal: intended findings and discussion prompts</strong></summary>

The dataset deliberately contains:

- missing `yaw_angle_deg` in **WT012**, `ride_height_mm` in **WT027**, and target `drag_coefficient` in **WT038**;
- an exact duplicated **WT021** record;
- likely unit/typing errors: **WT019** has 250 m/s, **WT033** has 420 mm ride height, **WT041** has $C_d=2.910$, and **WT044** has negative air density;
- **WT046** has a plausible-looking but relationship-breaking drag value. It is easy to miss with range checks alone.

Do not automatically delete every flagged row. A real extreme operating condition may be valuable. Check test logs, units, sensor calibration, and entry records first. A missing target cannot train a supervised model, while a missing input might be recovered or imputed later.
</details>

In [ ]:
# Instructor/check cell: assemble all deliberately planted records.
issue_mask = (
    df.isna().any(axis=1)
    | df.duplicated(keep=False)
    | ~df['wind_speed_m_s'].between(15, 70)
    | ~df['ride_height_mm'].between(70, 200)
    | ~df['air_density_kg_m3'].between(0.9, 1.4)
    | ~df['drag_coefficient'].between(0.20, 0.80)
)

display(df.loc[issue_mask].sort_values('test_id'))
print('Remember: WT046 is not found by simple range checks; the plots reveal it.')